In [1]:
import os
import math
import json
import torch
import random
import torchaudio
import torchaudio.transforms as T
import numpy as np
from tqdm import tqdm
from torch import nn, optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
from sklearn.metrics import *
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
W2V_CKPT_PATH = "/kaggle/input/models/rudranshiverma/w2v-audio-v2/pytorch/default/1/audio_w2v (1).pth"
 
checkpoint = torch.load(W2V_CKPT_PATH, map_location="cpu", weights_only=False)
print(f"Checkpoint epoch: {checkpoint['epoch']}")
print(f"Dev PR-AUC: {checkpoint['pr_auc']:.4f}")
print(f"Dev EER:    {checkpoint['eer']:.4f}")

Checkpoint epoch: 10
Dev PR-AUC: 1.0000
Dev EER:    0.0086


In [3]:
TARGET_SR = 16_000
MAX_LEN= TARGET_SR * 4
SEED= 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
noise_dir = "/kaggle/input/datasets/rudranshiverma/noise-dataset/audio"
noise_files = [os.path.join(noise_dir, f) for f in os.listdir(noise_dir) if f.endswith(".wav")]

def _fit_len(w):
    if w.shape[1] > MAX_LEN:
        return w[:, :MAX_LEN]
    return torch.nn.functional.pad(w, (0, MAX_LEN - w.shape[1]))

def augment_waveform(wav: torch.Tensor) -> torch.Tensor:
    p = random.random()

    #real noise
    if p < 0.20 and len(noise_files) > 0:
        npath = random.choice(noise_files)
        noise, sr = torchaudio.load(npath)

        if noise.shape[0] > 1:
            noise = noise.mean(dim=0, keepdim=True)

        if sr != TARGET_SR:
            noise = T.Resample(sr, TARGET_SR)(noise)

        noise = _fit_len(noise)

        sig_pow = wav.pow(2).mean()
        noise_pow = noise.pow(2).mean() + 1e-8

        snr_db = random.uniform(10, 20)
        scale = torch.sqrt(sig_pow / (10 ** (snr_db / 10) * noise_pow))

        wav = wav + scale * noise

    #synthetic light noise
    elif p < 0.35:
        wav = wav + 0.003 * torch.randn_like(wav)

    #gain
    if random.random() < 0.15:
        wav = wav * random.uniform(0.85, 1.15)

    return wav.clamp(-1, 1)

In [5]:
class W2VUnifiedDataset(Dataset):

    def __init__(
        self,
        protocol_path,
        flac_dir,
        extra_files=None,
        extra_labels=None,
        extra_sources=None,
        augment=False,
    ):
        self.samples = []
        self.augment = augment

        with open(protocol_path) as f:
            for line in f:
                parts = line.strip().split()
                fname = parts[1]
                label = 0 if parts[4] == "bonafide" else 1

                self.samples.append({
                    "path": os.path.join(flac_dir, fname + ".flac"),
                    "label": label,
                    "source": "asv"
                })

        if extra_files:
            for p, y, s in zip(extra_files, extra_labels, extra_sources):
                self.samples.append({
                    "path": p,
                    "label": y,
                    "source": s
                })

    def __len__(self):
        return len(self.samples)

    def _load_waveform(self, path):
        wav, sr = torchaudio.load(path)

        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        if sr != TARGET_SR:
            wav = T.Resample(sr, TARGET_SR)(wav)

        if wav.shape[1] > MAX_LEN:
            wav = wav[:, :MAX_LEN]
        else:
            wav = torch.nn.functional.pad(wav, (0, MAX_LEN - wav.shape[1]))

        return wav.squeeze(0)

    def __getitem__(self, idx):
        item = self.samples[idx]

        wav = self._load_waveform(item["path"])

        if self.augment:
            wav = augment_waveform(wav.unsqueeze(0)).squeeze(0)

        return wav.float(), torch.tensor(item["label"], dtype=torch.float32)

In [6]:
train_protocol = "/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
dev_protocol   = "/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt"
eval_protocol  = "/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt"

train_flac_dir = "/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_train/flac"
dev_flac_dir   = "/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_dev/flac"
eval_flac_dir  = "/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_eval/flac"

tts_dir = "/kaggle/input/datasets/rudranshiverma/modern-tts-dataset/Fake_ElevenLabs_Respeecher"
tts_files = [os.path.join(tts_dir, f) for f in os.listdir(tts_dir) if f.endswith(".wav")]
tts_labels = [1] * len(tts_files)
tts_sources = ["tts"] * len(tts_files)

libri_dir = "/kaggle/input/datasets/rudranshiverma/librispeech-subset"
libri_files = [os.path.join(libri_dir, f) for f in os.listdir(libri_dir) if f.endswith(".wav")]
libri_labels = [0] * len(libri_files)
libri_sources = ["libri"] * len(libri_files)

print("TTS:", len(tts_files))
print("Libri:", len(libri_files))

TTS: 600
Libri: 1000


In [7]:
extra_all = list(zip(
    tts_files + libri_files,
    tts_labels + libri_labels,
    tts_sources + libri_sources
))
random.shuffle(extra_all)
split = int(0.8 * len(extra_all))

extra_train = extra_all[:split]
extra_val= extra_all[split:]

extra_train_files= [x[0] for x in extra_train]
extra_train_labels= [x[1] for x in extra_train]
extra_train_sources = [x[2] for x in extra_train]

extra_val_files= [x[0] for x in extra_val]
extra_val_labels= [x[1] for x in extra_val]
extra_val_sources = [x[2] for x in extra_val]

In [8]:
train_dataset = W2VUnifiedDataset(
    train_protocol,
    train_flac_dir,
    extra_train_files,
    extra_train_labels,
    extra_train_sources,
    augment=True
)

dev_dataset = W2VUnifiedDataset(
    dev_protocol,
    dev_flac_dir,
    augment=False
)

extra_val_dataset = W2VUnifiedDataset(
    dev_protocol,
    dev_flac_dir,
    extra_val_files,
    extra_val_labels,
    extra_val_sources,
    augment=False
)

extra_val_dataset.samples = [
    s for s in extra_val_dataset.samples if s["source"] != "asv"
]

print("Train:", len(train_dataset))
print("Dev:", len(dev_dataset))
print("Extra val:", len(extra_val_dataset))

Train: 26660
Dev: 24844
Extra val: 320


In [10]:
sample_weights = []

for s in train_dataset.samples:
    src = s["source"]
    y= s["label"]

    if src == "asv" and y == 0:
        w = 1/3
    elif src == "asv" and y == 1:
        w = 1/3
    elif src == "tts":
        w = 1/6
    elif src == "libri":
        w = 1/6
    else:
        w = 1.0

    sample_weights.append(w)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    sampler=sampler,
    num_workers=2,
    pin_memory=True,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

extra_val_loader = DataLoader(
    extra_val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

In [11]:
class W2VModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

        for name, param in self.encoder.named_parameters():
            if "encoder.layers.10" not in name and "encoder.layers.11" not in name:
                param.requires_grad = False

        self.attention = nn.Sequential(
            nn.Linear(768, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )

        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, wav):
        wav = (wav - wav.mean(dim=-1, keepdim=True)) / (
            wav.std(dim=-1, keepdim=True) + 1e-7
        )

        hidden = self.encoder(wav).last_hidden_state
        attn = torch.softmax(self.attention(hidden), dim=1)
        pooled = torch.sum(hidden * attn, dim=1)

        return self.classifier(pooled)

model = W2VModel().to(device)
model.load_state_dict(checkpoint["model_state_dict"])
print("Loaded.")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loaded.


In [12]:
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=5e-6,
    weight_decay=1e-4
)
EPOCHS = 10
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS * len(train_loader),
    eta_min=1e-7
)

In [13]:
def hmean(a, b):
    return 2*a*b/(a+b+1e-8)

def evaluate(model, loader, loader_name=""):
    model.eval()

    all_scores = []
    all_labels = []

    with torch.no_grad():
        for wav, labels in loader:
            wav = wav.to(device)
            scores = torch.sigmoid(model(wav)).view(-1)

            all_scores.extend(scores.cpu().numpy())
            all_labels.extend(labels.numpy())

    scores_np = np.array(all_scores)
    labels_np = np.array(all_labels)

    pr_auc  = average_precision_score(labels_np, scores_np)
    roc_auc = roc_auc_score(labels_np, scores_np)

    fpr, tpr, _ = roc_curve(labels_np, scores_np)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.abs(fnr-fpr))]

    prec_arr, rec_arr, thresh_arr = precision_recall_curve(labels_np, scores_np)

    valid = np.where(rec_arr[:-1] >= 0.90)[0]

    if len(valid) > 0:
        best_thresh = thresh_arr[valid[np.argmax(prec_arr[valid])]]
    else:
        best_thresh = 0.5

    preds = (scores_np > best_thresh).astype(int)

    rec= recall_score(labels_np, preds, zero_division=0)
    prec= precision_score(labels_np, preds, zero_division=0)
    f1= f1_score(labels_np, preds, zero_division=0)
    acc= accuracy_score(labels_np, preds)
    cm= confusion_matrix(labels_np, preds)

    missed = int(cm[1][0])

    print(
        f"[{loader_name}] "
        f"PR-AUC={pr_auc:.4f} "
        f"ROC-AUC={roc_auc:.4f} "
        f"EER={eer:.4f} "
        f"Recall={rec:.4f} "
        f"Missed={missed}"
    )

    return {
        "pr_auc": float(pr_auc),
        "roc_auc": float(roc_auc),
        "eer": float(eer),
        "recall": float(rec),
        "precision": float(prec),
        "f1": float(f1),
        "accuracy": float(acc),
        "threshold": float(best_thresh),
        "missed": missed,
    }

In [ ]:
best_score = 0.0
best_eer = 1.0
patience = 4
no_improve = 0
history = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for wav, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        wav = wav.to(device)
        labels = labels.unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(wav)
        loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    print(f"\nEpoch {epoch+1} train_loss={avg_loss:.5f}")

    dev_metrics = evaluate(model, dev_loader, "ASVspoof-dev")
    extra_metrics = evaluate(model, extra_val_loader, "Extra-val")

    score = hmean(
        extra_metrics["pr_auc"],
        dev_metrics["recall"]
    )

    guard = (
        dev_metrics["eer"] <= 0.12 and
        dev_metrics["recall"] >= 0.90 and
        extra_metrics["recall"] >= 0.85
    )

    history.append({
        "epoch": epoch+1,
        "train_loss": avg_loss,
        "dev": dev_metrics,
        "extra_val": extra_metrics,
        "selection_score": score
    })

    if dev_metrics["eer"] < best_eer:
        best_eer = dev_metrics["eer"]

    if guard and score > best_score:
        best_score = score
        no_improve = 0

        torch.save({
            "epoch": epoch+1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "dev_metrics": dev_metrics,
            "extra_val_metrics": extra_metrics,
            "selection_score": score,
            "best_dev_eer_seen": best_eer,
        }, "/kaggle/working/w2v_finetuned.pth")

        print("✓ Best model saved")

    else:
        no_improve += 1
        print("No improvement", no_improve)

        if no_improve >= patience:
            print("Early stopping")
            break

with open("/kaggle/working/w2v_finetune_history.json", "w") as f:
    json.dump(history, f, indent=2)

In [15]:
best_ckpt = torch.load(
    "/kaggle/input/models/rudranshiverma/w2v2-model-v2/pytorch/default/1/w2v_finetuned.pth",
    map_location=device,
    weights_only=False
)

model.load_state_dict(best_ckpt["model_state_dict"])
model.eval()

eval_dataset = W2VUnifiedDataset(
    eval_protocol,
    eval_flac_dir,
    augment=False
)
eval_loader = DataLoader(
    eval_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)
print("\n=== FINAL EVAL — ASVspoof eval ===")
eval_metrics = evaluate(model, eval_loader, "ASVspoof-eval")

with open("/kaggle/working/w2v_eval_final.json", "w") as f:
    json.dump(eval_metrics, f, indent=2)

print("Saved final eval.")


=== FINAL EVAL — ASVspoof eval ===
[ASVspoof-eval] PR-AUC=0.9983 ROC-AUC=0.9854 EER=0.0408 Recall=0.9001 Missed=6385
Saved final eval.
